In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/software-build-failure-prediction/data/processed/travistorrent_ruby_builds.csv')
print(df.shape)
df.head()

/tmp/ipykernel_3768/1014628402.py:3: DtypeWarning: Columns (3,6,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/software-build-failure-prediction/data/processed/travistorrent_ruby_builds.csv')


(252151, 66)


,tr_build_id,gh_project_name,gh_is_pr,gh_pr_created_at,gh_pull_req_num,gh_lang,git_merged_with,git_branch,gh_num_commits_in_push,gh_commits_in_push,...,tr_original_commit,tr_duration,tr_status,tr_jobs,tr_log_num_tests_failed,tr_log_num_tests_ok,tr_log_num_tests_run,tr_log_bool_tests_failed,tr_log_testduration,tr_log_buildduration
0,3154,rspec/rspec-core,False,NaN,NaN,ruby,NaN,master,NaN,NaN,...,029e6972fcf719542deff1b2619d2945146e84da,956.0,passed,"[3161, 3163, 3160, 3162, 3164]",0.0,4000.0,4000.0,False,758.53,0.0
1,3211,ruby-amqp/amqp,False,NaN,NaN,ruby,NaN,master,NaN,NaN,...,7fc267b949209b189fa76f232f7e88a9b938b363,391.0,passed,"[3214, 3215, 3212, 3216, 3213]",0.0,410.0,410.0,False,159.32,0.0
2,3233,rspec/rspec-core,False,NaN,NaN,ruby,NaN,master,NaN,NaN,...,36237841db72eeb7797f199ad0fa057e36e26b40,886.0,failed,"[3235, 3234, 3238, 3237, 3236]",5.0,4040.0,4045.0,True,693.85,0.0
3,3239,rspec/rspec-core,False,NaN,NaN,ruby,NaN,master,NaN,NaN,...,03a13bcdf21eeae059b2c535338465c7b212f126,846.0,passed,"[3244, 3240, 3242, 3243, 3241]",0.0,4005.0,4005.0,False,697.15,0.0
4,3245,rspec/rspec-core,False,NaN,NaN,ruby,NaN,master,NaN,NaN,...,87a0f3890473357918b0b88753094d7a84893d71,846.0,passed,"[3246, 3250, 3248, 3249, 3247]",0.0,4005.0,4005.0,False,694.37,0.0


In [3]:
PRE_EXECUTION_FEATURES = [
    'git_branch', 'gh_num_commits_in_push', 'git_prev_commit_resolution_status',
    'gh_is_pr', 'gh_pull_req_num', 'gh_lang', 'gh_first_commit_created_at', 'gh_team_size',
    'gh_num_issue_comments', 'gh_num_commit_comments', 'gh_num_pr_comments',
    'git_diff_src_churn', 'git_diff_test_churn', 'gh_diff_files_added', 'gh_diff_files_deleted',
    'gh_diff_files_modified', 'gh_diff_tests_added', 'gh_diff_tests_deleted', 'gh_diff_src_files',
    'gh_diff_doc_files', 'gh_diff_other_files', 'gh_num_commits_on_files_touched', 'gh_sloc',
    'gh_test_lines_per_kloc', 'gh_test_cases_per_kloc', 'gh_asserts_cases_per_kloc',
    'gh_by_core_team_member', 'gh_description_complexity', 'gh_pushed_at', 'gh_repo_age',
    'gh_repo_num_commits', 'tr_build_number', 'gh_build_started_at'
]

POST_EXECUTION_FEATURES = [
    'tr_log_status', 'tr_log_setup_time', 'tr_log_analyzer', 'tr_log_frameworks',
    'tr_log_bool_tests_ran', 'tr_log_bool_tests_failed', 'tr_log_num_tests_ok',
    'tr_log_num_tests_failed', 'tr_log_num_tests_run', 'tr_log_num_tests_skipped',
    'tr_log_num_test_suites_run', 'tr_log_num_test_suites_ok', 'tr_log_num_test_suites_failed',
    'tr_log_tests_failed', 'tr_log_testduration', 'tr_log_buildduration', 'tr_duration', 'tr_log_lan'
]

DROP_COLUMNS = ['tr_build_id', 'gh_project_name', 'git_merged_with', 'tr_virtual_merged_into',
                'tr_job_id', 'tr_original_commit', 'tr_jobs']

print('Pre-execution features:', len(PRE_EXECUTION_FEATURES))
print('Post-execution features:', len(POST_EXECUTION_FEATURES))

Pre-execution features: 33
Post-execution features: 18


In [4]:
df = df[df['tr_status'].isin(['passed', 'failed'])].copy()
df['target'] = (df['tr_status'] == 'failed').astype(int)

print(df['target'].value_counts())

target
0    197850
1     54301
Name: count, dtype: int64


In [5]:
df_clean = df[list(dict.fromkeys(PRE_EXECUTION_FEATURES + ['target']))].copy()
df_leaky = df[list(dict.fromkeys(PRE_EXECUTION_FEATURES + POST_EXECUTION_FEATURES + ['target']))].copy()

print('Clean pipeline shape:', df_clean.shape)
print('Leaky pipeline shape:', df_leaky.shape)

Clean pipeline shape: (252151, 34)
Leaky pipeline shape: (252151, 52)


In [6]:
fully_empty_cols = ['gh_num_commits_in_push', 'gh_pushed_at', 'gh_first_commit_created_at']

df_clean = df_clean.drop(columns=[c for c in fully_empty_cols if c in df_clean.columns])
df_leaky = df_leaky.drop(columns=[c for c in fully_empty_cols if c in df_leaky.columns])

print('Clean shape after drop:', df_clean.shape)
print('Leaky shape after drop:', df_leaky.shape)

Clean shape after drop: (252151, 31)
Leaky shape after drop: (252151, 49)


In [7]:
print("Missing % in df_clean:")
print(df_clean.isnull().mean().sort_values(ascending=False) * 100)
print("\nMissing % in df_leaky:")
print(df_leaky.isnull().mean().sort_values(ascending=False) * 100)

Missing % in df_clean:
gh_description_complexity            81.596345
gh_num_issue_comments                81.307233
gh_pull_req_num                      81.307233
gh_num_pr_comments                   81.307233
gh_is_pr                              0.000000
gh_lang                               0.000000
git_prev_commit_resolution_status     0.000000
gh_team_size                          0.000000
gh_num_commit_comments                0.000000
git_diff_src_churn                    0.000000
git_diff_test_churn                   0.000000
gh_diff_files_added                   0.000000
gh_diff_files_deleted                 0.000000
gh_diff_files_modified                0.000000
git_branch                            0.000000
gh_diff_tests_added                   0.000000
gh_diff_tests_deleted                 0.000000
gh_diff_doc_files                     0.000000
gh_diff_src_files                     0.000000
gh_num_commits_on_files_touched       0.000000
gh_sloc                              

In [8]:
for col in ['gh_pull_req_num', 'gh_num_issue_comments', 'gh_num_pr_comments']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(0)

numeric_cols = df_clean.select_dtypes(include='number').columns
for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

categorical_cols = df_clean.select_dtypes(include='object').columns
for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna('unknown')

print(df_clean.isnull().sum().sum(), 'missing values remaining in df_clean')

0 missing values remaining in df_clean


In [9]:
for col in ['gh_pull_req_num', 'gh_num_issue_comments', 'gh_num_pr_comments']:
    if col in df_leaky.columns:
        df_leaky[col] = df_leaky[col].fillna(0)

numeric_cols = df_leaky.select_dtypes(include='number').columns
for col in numeric_cols:
    df_leaky[col] = df_leaky[col].fillna(df_leaky[col].median())

categorical_cols = df_leaky.select_dtypes(include='object').columns
for col in categorical_cols:
    df_leaky[col] = df_leaky[col].fillna('unknown')

print(df_leaky.isnull().sum().sum(), 'missing values remaining in df_leaky')

0 missing values remaining in df_leaky


In [10]:
print('Final df_clean shape:', df_clean.shape)
print('Final df_leaky shape:', df_leaky.shape)

df_clean.to_csv('/content/drive/MyDrive/software-build-failure-prediction/data/processed/df_clean.csv', index=False)
df_leaky.to_csv('/content/drive/MyDrive/software-build-failure-prediction/data/processed/df_leaky.csv', index=False)
print('Both files saved.')

Final df_clean shape: (252151, 31)
Final df_leaky shape: (252151, 49)
Both files saved.
